# Kiểm tra lại môi trường Kaggle offline

Mở **phiên Kaggle mới**, chọn GPU dự định dùng, đặt **Internet Off** rồi **Run All**.
Không cần gắn Dataset packages và không chạy setup notebook 3 trước lần kiểm tra này.

Notebook chỉ đọc thông tin, import thư viện và thử tensor/ảnh giả rất nhỏ.
**Không cài packages, không tải weights, không chạy OCR.** Hunyuan có timeout import riêng 180 giây.
Metadata, lỗi import, timeout và kết quả GPU được báo riêng; package có metadata chưa chắc dùng được,
package thiếu cũng không có nghĩa phải cài lại toàn bộ dependency.


In [ ]:
"""Read-only environment inventory; no installs, downloads, weights or OCR."""
from __future__ import annotations

from datetime import datetime, timezone
import importlib.metadata as metadata
import importlib.util
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import sysconfig


PACKAGES = (
    "torch", "torchvision", "transformers", "tokenizers", "accelerate", "safetensors",
    "huggingface-hub", "sentencepiece", "protobuf", "numpy", "scipy", "pillow", "opencv-python", "opencv-python-headless",
    "paddlepaddle", "paddlepaddle-gpu", "paddleocr", "paddlex", "mmcv", "mmcv-full",
    "mmcv-lite", "mmengine", "mmdet", "mmrotate", "timm", "einops", "flash-attn",
    "triton", "pyyaml", "pypdfium2", "pandas", "tqdm", "pip", "setuptools", "wheel",
    "packaging", "ninja", "cmake", "uv",
)

PROBES = {
    "torch_cuda": ("torch", 90, """import torch, json
print('stage: torch imported', flush=True)
devices = [{'name': torch.cuda.get_device_name(i), 'VRAM_GiB': round(torch.cuda.get_device_properties(i).total_memory/1024**3,2), 'compute_capability': list(torch.cuda.get_device_capability(i))} for i in range(torch.cuda.device_count())]
result = {'torch': torch.__version__, 'built_cuda': torch.version.cuda, 'cuda_available': torch.cuda.is_available(), 'devices': devices}
if torch.cuda.is_available():
    x = torch.ones((2,2), device='cuda:0')
    result['gpu_matmul_sum'] = float((x @ x).sum().item())
print(json.dumps(result))
"""),
    "torchvision_ops": ("torchvision", 90, """import torch, torchvision, json
from torchvision.ops import nms
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
boxes = torch.tensor([[0.,0.,2.,2.],[0.,0.,2.,2.]],device=device)
keep = nms(boxes, torch.tensor([.9,.8],device=device), .5)
assert keep.numel() == 1
print(json.dumps({'torchvision': torchvision.__version__, 'nms_device': device}))
"""),
    "torch_numpy_bridge": ("torch", 45, """import torch, numpy as np, json
x = torch.from_numpy(np.zeros((2,2),dtype=np.float32))
assert x.shape == (2,2) and x.numpy().shape == (2,2)
print(json.dumps({'torch':torch.__version__,'numpy':np.__version__,'numpy_bridge':'OK'}))
"""),
    "numpy_cv2": ("cv2", 45, """import numpy as np, cv2, json
from PIL import Image
x = np.zeros((4,4,3),dtype=np.uint8)
assert cv2.cvtColor(x,cv2.COLOR_RGB2GRAY).shape == (4,4)
assert Image.fromarray(x).size == (4,4)
print(json.dumps({'numpy': np.__version__, 'opencv': cv2.__version__, 'array_image_bridge': 'OK'}))
"""),
    "transformers_hunyuan": ("transformers", 180, """import json
print('stage: transformers import', flush=True)
import transformers
print('stage: AutoProcessor', flush=True)
from transformers import AutoProcessor
print('stage: Hunyuan class', flush=True)
from transformers import HunYuanVLForConditionalGeneration
print(json.dumps({'transformers': transformers.__version__, 'hunyuan_api': 'OK'}))
"""),
    "paddle_cuda": ("paddle", 90, """import paddle, json
result = {'paddle': paddle.__version__, 'compiled_with_cuda': paddle.is_compiled_with_cuda()}
result['devices'] = paddle.device.cuda.device_count() if result['compiled_with_cuda'] else 0
if result['devices']:
    paddle.set_device('gpu:0')
    x = paddle.ones((2,2))
    result['gpu_matmul_sum'] = float(paddle.matmul(x,x).sum().item())
print(json.dumps(result))
"""),
    "paddleocr_apis": ("paddleocr", 120, """import paddleocr, json
from paddleocr import PaddleOCR, PaddleOCRVL
print(json.dumps({'paddleocr': paddleocr.__version__, 'PaddleOCR': 'OK', 'PaddleOCRVL': 'OK'}))
"""),
    "mmcv_compiled_ops": ("mmcv", 120, """import torch, mmcv, mmcv.ops, mmdet, mmrotate, json
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
boxes = torch.tensor([[5.,5.,2.,2.,0.]],device=device)
scores = torch.tensor([.9],device=device)
_, keep = mmcv.ops.nms_rotated(boxes,scores,.5)
assert keep.numel() == 1
print(json.dumps({'mmcv': mmcv.__version__, 'mmdet': mmdet.__version__, 'mmrotate': mmrotate.__version__, 'rotated_nms_device': device}))
"""),
    "pdf_renderer": ("pypdfium2", 45, """import pypdfium2 as pdfium, json
doc = pdfium.PdfDocument.new()
page = doc.new_page(32,32)
bitmap = page.render(scale=1)
image = bitmap.to_pil()
assert image.size == (32,32)
image.close(); bitmap.close(); page.close(); doc.close()
print(json.dumps({'blank_page_32x32': 'OK'}))
"""),
}


def run_command(command, timeout=30):
    env = {**os.environ, "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
           "HF_HUB_DISABLE_TELEMETRY": "1", "PIP_NO_INDEX": "1", "USE_TF": "0", "USE_FLAX": "0",
           "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK": "True"}
    try:
        result = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", errors="replace",
                                timeout=timeout, env=env)
        return {"status": "ok" if result.returncode == 0 else "failed", "returncode": result.returncode,
                "stdout": result.stdout, "stderr": result.stderr}
    except subprocess.TimeoutExpired as exc:
        def decode(value):
            return value.decode("utf-8", errors="replace") if isinstance(value, bytes) else value or ""
        return {"status": "timeout", "timeout_seconds": timeout,
                "stdout": decode(exc.stdout), "stderr": decode(exc.stderr)}
    except OSError as exc:
        return {"status": "unavailable", "detail": str(exc)}


def read_text(path):
    try:
        return Path(path).read_text(encoding="utf-8").strip()
    except OSError:
        return None


def hardware():
    meminfo = {}
    for line in (read_text("/proc/meminfo") or "").splitlines():
        key, value = line.split(":", 1)
        meminfo[key] = int(value.strip().split()[0]) * 1024
    limit = read_text("/sys/fs/cgroup/memory.max") or read_text("/sys/fs/cgroup/memory/memory.limit_in_bytes")
    current = read_text("/sys/fs/cgroup/memory.current") or read_text("/sys/fs/cgroup/memory/memory.usage_in_bytes")
    limit = int(limit) if limit and limit.isdigit() else None
    current = int(current) if current and current.isdigit() else None
    totals = [value for value in (meminfo.get("MemTotal"), limit) if value is not None]
    total = min(totals) if totals else None
    available = meminfo.get("MemAvailable")
    if limit is not None and current is not None:
        available = min(available if available is not None else limit, max(0, limit-current))
    quota = read_text("/sys/fs/cgroup/cpu.max")
    try:
        affinity = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None
    except OSError:
        affinity = None
    cpu_model = next((line.split(":",1)[1].strip() for line in (read_text("/proc/cpuinfo") or "").splitlines()
                      if line.startswith("model name")), platform.processor())
    disks = {}
    for name in ("/", "/kaggle/input", "/kaggle/working", "/tmp"):
        if Path(name).is_dir():
            disk = shutil.disk_usage(name)
            disks[name] = {"total_GiB": round(disk.total/1024**3,2), "free_GiB": round(disk.free/1024**3,2)}
    return {"os": platform.platform(), "machine": platform.machine(), "libc": platform.libc_ver(),
            "python": sys.version, "python_executable": sys.executable, "python_ABI": sysconfig.get_config_var("SOABI"),
            "cpu_model": cpu_model, "cpu_logical": os.cpu_count(), "cpu_affinity": affinity, "cpu_quota_raw": quota,
            "RAM_effective_GiB": round(total/1024**3,2) if total is not None else None,
            "RAM_available_GiB": round(available/1024**3,2) if available is not None else None, "disks": disks}


def distributions():
    records = []
    seen = set()
    for dist in metadata.distributions():
        name = dist.metadata.get('Name', '')
        key = name.lower().replace('_', '-').replace('.', '-')
        records.append({"name": name, "version": dist.version, 'active': key not in seen,
                        'location': str(dist.locate_file('')),
                        "requires_python": dist.metadata.get("Requires-Python"), "requires": dist.requires or []})
        seen.add(key)
    return sorted(records, key=lambda item: item["name"].lower())


def offline_inputs(root=Path("/kaggle/input"), max_dirs=150, max_files=80):
    found, visited = [], 0
    if not root.is_dir():
        return {"exists": False, "files": [], "truncated": False}
    for parent, dirs, files in os.walk(root):
        visited += 1
        if len(Path(parent).relative_to(root).parts) >= 6:
            dirs[:] = []
        for name in sorted(files):
            if name.endswith((".whl", ".tar.gz", ".tar.xz", ".zip", ".safetensors", ".pth")) or name in {"config.json", "inference.yml", "bundle_manifest.json"}:
                path = Path(parent) / name
                try:
                    size = path.stat().st_size
                except OSError:
                    continue
                found.append({"path": str(path), "bytes": size})
                if len(found) >= max_files:
                    return {"exists": True, "files": found, "truncated": True, "visited_dirs": visited}
        if visited >= max_dirs:
            return {"exists": True, "files": found, "truncated": True, "visited_dirs": visited}
    return {"exists": True, "files": found, "truncated": False, "visited_dirs": visited}


def collect_environment(*, run_probes=True):
    inventory = distributions()
    versions = {}
    for item in inventory:
        name = item['name'].lower().replace('_', '-')
        if name not in versions or item.get('active') is True:
            versions[name] = item['version']
    report = {"schema": 1, "collected_at_UTC": datetime.now(timezone.utc).isoformat(),
              "hardware": hardware(), "packages": {name: versions.get(name) for name in PACKAGES},
              "all_distributions": inventory, "probes": {}, "tools": {}, "offline_inputs": offline_inputs(),
              "prior_vtm_runtime_exists": Path("/tmp/vtm_kaggle_runtime/runtime_config.json").is_file(),
              "note": "Read-only diagnostics. Imports/tiny ops only; no installation, downloads, weights or OCR."}
    if not run_probes:
        return report
    commands = {"gpu": ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free,driver_version", "--format=csv,noheader"],
                "nvcc": ["nvcc", "--version"], "g++": ["g++", "--version"],
                "cmake": ["cmake", "--version"], "ninja": ["ninja", "--version"]}
    for name, command in commands.items():
        report["tools"][name] = run_command(command)
    report["pip_check"] = run_command([sys.executable, "-m", "pip", "check"], timeout=60)
    for name, (module, timeout, code) in PROBES.items():
        print(f"Checking {name} (timeout {timeout}s)...", flush=True)
        try:
            present = importlib.util.find_spec(module) is not None
        except (ImportError, ValueError):
            present = False
        result = run_command([sys.executable, "-c", code], timeout) if present else {"status": "not_installed"}
        report["probes"][name] = result
        print(f"  {result['status']}", flush=True)
    return report


def save_report(report, output_dir):
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    json_path = output / "kaggle_environment_report.json"
    text_path = output / "kaggle_environment_report.txt"
    json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    lines = [json.dumps(report["hardware"], ensure_ascii=False, indent=2), "\nPackages:"]
    lines.extend(f"{name:24s} {version or 'not installed'}" for name, version in report["packages"].items())
    for group in ("tools", "probes"):
        for name, result in report[group].items():
            lines.append(f"\n[{name}]\n" + json.dumps(result, ensure_ascii=False, indent=2))
    lines.append("\n[pip_check]\n" + json.dumps(report.get("pip_check"), ensure_ascii=False, indent=2))
    text_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return json_path, text_path




## 1. Kiểm tra CPU/RAM/GPU, thư viện và các phép thử nhỏ


In [ ]:
report = collect_environment()
print(json.dumps(report["hardware"], ensure_ascii=False, indent=2))
print()
print("GPU:", report["tools"].get("gpu", {}).get("stdout", "unavailable"))
print("Torch/CUDA:", report["probes"].get("torch_cuda", {}).get("stdout", "not available"))
print()
print("Các package liên quan:")
for name, version in report["packages"].items():
    print(f"{name:24s} {version or 'not installed'}")
print()
print("Pip dependency check:", report.get("pip_check", {}).get("status"))
print("Runtime VTM từ phiên trước:", report["prior_vtm_runtime_exists"])


## 2. Lưu báo cáo để chốt lại packages còn thiếu

Gửi lại **`kaggle_environment_report.json`** trong Output. File TXT chứa log dễ đọc.
JSON gồm toàn bộ phiên bản/dependency đang có; dùng nó để chọn phần cần bổ sung,
thay vì mặc định gom toàn bộ thư viện vào môi trường riêng.


In [ ]:
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("work/kaggle_environment_check")
json_path, text_path = save_report(report, OUTPUT_DIR)
print("Báo cáo JSON:", json_path)
print("Báo cáo TXT:", text_path)
